In [21]:
from pathlib import Path
import pandas as pd

In [34]:
csv_file = Path.home()/'Desktop/LakeViewHeadings2.csv'

headings = pd.read_csv(csv_file)

In [71]:
from enspec.database import EnSpecDB

db = EnSpecDB()

In [76]:
db.Sessions[['flight_id', 'session_id']].merge(db.Lines[db.Lines.line_id.isin(exclude_lines)], on='session_id').merge(db.Flights[['flight_id', 'date']], on='flight_id')

,flight_id,session_id,line_id,line_number,line_index,time_start,time_end,quality,notes,date
0,dbed53aa-b776-4994-a25d-2df5b1733d27,39237b26-1575-4d4a-987a-ac8cda1fe0f0,dd7433f8-37d8-4447-8935-dbd9a022be7e,1,NaN,NaT,NaT,RED,Accidental image on tarmac.,2024-05-17
1,a96d29a3-fba3-4078-afdf-394e5dd61775,8d6164a3-c713-4df7-8209-71238f4710eb,ed5e5359-b732-46ce-b22f-a28620a45adc,1,NaN,NaT,NaT,GREEN,Self-portrait taken on tarmac.,2024-09-25
2,b0918c85-b074-4415-ad93-df2232c4a38e,83cea4ea-68aa-42b1-8776-3285e40ad720,a0340c7c-5dda-48de-b9de-d604cf8d5011,14,NaN,NaT,NaT,RED,Very short image collected during aborted pass...,2024-10-07


In [ ]:
exclude = headings.heading.isin([30, 40, 60, 150])
exclude_lines = list(set(headings[exclude].line_id.values))

headings = headings[~exclude]

# Manual corrections (LakeView 2024)
headings.loc[headings.heading == 130, 'heading'] = 120

In [35]:
sorted(set(headings.heading.values))

[20.0, 110.0, 120.0, 140.0, 170.0]

In [27]:
joblist_file = Path(r"Z:\users\bheberlein\processing\2024\LAKE\raw-inputs\LAKE_2024-LAKE_JobList.txt")

In [32]:
joblist = pd.read_csv(joblist_file, header=None, names=['session_name', 'isodate', 'line_number', 'disk', 'memory'])

In [38]:
from enspec.display.table import pretty_print

In [53]:
len(swir_headings)

306

In [52]:
swir_images = headings.filepath.apply(lambda p: '_SWIR_384_' in p)
swir_headings = headings[swir_images]

In [61]:
from_headings = sorted(set([tuple(map(str, v)) for v in swir_headings[['session_name', 'isodate', 'line_number']].values]))

In [62]:
from_joblist = sorted(set([tuple(map(str, v)) for v in joblist[['session_name', 'isodate', 'line_number']].values]))

In [64]:
len(from_joblist), len(from_headings)

(329, 306)

In [83]:
# NOTE: Some images are from 2023; others just don't need to be processed
[job for job in from_joblist if job not in from_headings and not job[1][:4] == '2023']

[('LAKE', '20240517', '1'),
 ('LAKE', '20240925', '1'),
 ('LAKE', '20241007', '14')]

In [82]:
len(_)

23

In [67]:
[job for job in from_headings if job not in from_joblist]

[]

In [46]:
len(joblist)

329

In [49]:
headings[['session_name', 'isodate', 'line_number']]

,session_name,isodate,line_number
0,LAKE,20240315,1
1,LAKE,20240315,1
2,LAKE,20240315,2
3,LAKE,20240315,2
4,LAKE,20240315,3
...,...,...,...
607,LAKE,20241007,16
608,POND,20241007,1
609,POND,20241007,1
610,POND,20241007,2


In [86]:
merged = joblist.merge(swir_headings, on=['session_name', 'isodate', 'line_number'], how='inner')

In [96]:
output_directory = Path('Z:/users/bheberlein/processing/2024/LAKE/raw-inputs')

In [101]:
    print(site)
    print(heading)
    print(group)
    break

LAKE
110.0
    session_name   isodate  line_number   disk memory  heading
80          LAKE  20240517           16   36GB    9GB    110.0
214         LAKE  20240904            4   37GB    9GB    110.0


In [106]:
for (site, heading), group in merged[['session_name', 'isodate', 'line_number', 'disk', 'memory', 'heading']].groupby(['session_name', 'heading']):

    print(f'Working on group: {site} @{heading:.1f} ({len(group)})')
    joblist_file = output_directory/f'{site}_2024_H{int(heading)}_JobList.txt'
    with open(joblist_file, mode='w') as f:
        for v in group.drop(columns='heading').values:
            f.write(','.join(map(str, v))+'\n')

Working on group: LAKE @110.0 (2)
Working on group: LAKE @120.0 (125)
Working on group: LAKE @140.0 (111)
Working on group: LAKE @170.0 (33)
Working on group: LAKE-EXTRA @20.0 (1)
Working on group: LAKE-MISC @20.0 (1)
Working on group: LAKE-MISC @110.0 (1)
Working on group: POND @110.0 (1)
Working on group: POND @120.0 (15)
Working on group: POND @140.0 (16)


In [40]:
len(joblist)

329

In [26]:
headings

,session_id,line_id,image_id,session_name,line_number,heading,filepath
0,ca0578b9-9cb7-4c2e-afb8-c60af1618141,6e63bd37-d9ad-4a2e-9c4b-d470966a2d50,df5c0010-c14f-4b16-a448-ca2857d9b707,LAKE,1,170.0,Z:/data/collection/airborne/raw/2024/20240315/...
1,ca0578b9-9cb7-4c2e-afb8-c60af1618141,6e63bd37-d9ad-4a2e-9c4b-d470966a2d50,133d77ca-1c3b-49b0-963b-3b376440644b,LAKE,1,170.0,Z:/data/collection/airborne/raw/2024/20240315/...
2,ca0578b9-9cb7-4c2e-afb8-c60af1618141,f9c507da-cc4f-440d-88db-ac07af8f9625,c57e52d7-ec10-45b3-94a9-9f0acad5c3ca,LAKE,2,140.0,Z:/data/collection/airborne/raw/2024/20240315/...
3,ca0578b9-9cb7-4c2e-afb8-c60af1618141,f9c507da-cc4f-440d-88db-ac07af8f9625,ee75b05a-160b-4b27-a0b2-6323537ececa,LAKE,2,140.0,Z:/data/collection/airborne/raw/2024/20240315/...
4,ca0578b9-9cb7-4c2e-afb8-c60af1618141,89c6d248-3156-448f-a795-4da730382b5f,ba6277da-c8ad-454c-b9f4-39fa93255375,LAKE,3,140.0,Z:/data/collection/airborne/raw/2024/20240315/...
...,...,...,...,...,...,...,...
607,83cea4ea-68aa-42b1-8776-3285e40ad720,ed2acce0-2dd6-4805-b4d3-3cf5d10bd11e,0641904c-b746-4a1a-a806-cc43f632cde6,LAKE,16,170.0,Z:/data/collection/airborne/raw/2024/20241007/...
608,d3f55a97-2595-4678-ad94-15b1dc1da62a,62650be7-71c1-41c6-a52c-27394db47ebb,f0c4b24f-48ce-434a-a351-716cd8b0b7c5,POND,1,140.0,Z:/data/collection/airborne/raw/2024/20241007/...
609,d3f55a97-2595-4678-ad94-15b1dc1da62a,62650be7-71c1-41c6-a52c-27394db47ebb,583f432e-06a7-4315-bc6a-186c1d904a6c,POND,1,140.0,Z:/data/collection/airborne/raw/2024/20241007/...
610,d3f55a97-2595-4678-ad94-15b1dc1da62a,79323550-b6b1-49ae-9a53-80c9c395ee1e,b3270801-77b4-4d96-81c4-93484cd3a6dd,POND,2,140.0,Z:/data/collection/airborne/raw/2024/20241007/...
